# vmChat — Colab + Gemini 临时联调

这个 Notebook 只把 **Python AI 后端** 放到 Colab。前端继续在你的电脑本地运行，这样它可以同时访问：

- 本地业务接口：`http://192.168.1.203:8010/pflfofrest/`
- Colab Python：通过 Cloudflare Quick Tunnel 暴露的临时 HTTPS 地址
- Gemini：由 Colab Python 通过 Gemini OpenAI-compatible API 调用

这样不会把 Gemini API Key 提交到 GitHub。推荐把 Key 放在 Colab 左侧 **🔑 Secrets** 中，名称为 `GEMINI_API_KEY`。Google 官方也推荐在 Colab 中用 Secrets 保存 API Key。


In [ ]:
# 1) 拉取最新代码并安装 Python 依赖
!rm -rf /content/vmchat-hermes-replacement-python
!git clone -q https://github.com/liuruibing/vmchat-hermes-replacement-python.git /content/vmchat-hermes-replacement-python
%cd /content/vmchat-hermes-replacement-python
!python -m pip install -q uv
!uv sync --locked --no-dev --no-editable
print('✅ 代码与依赖准备完成')


In [ ]:
# 2) 读取 Gemini Key，并配置 vmChat
import os, secrets, getpass

try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except Exception:
    GEMINI_API_KEY = None

if not GEMINI_API_KEY:
    GEMINI_API_KEY = getpass.getpass('没有找到 Colab Secret GEMINI_API_KEY，请临时输入（不会回显）: ').strip()

if not GEMINI_API_KEY:
    raise RuntimeError('GEMINI_API_KEY 不能为空')

MODEL_ID = 'gemini-3.8-flash' # @param ['gemini-3.8-flash', 'gemini-3.5-flash-lite', 'gemini-2.5-flash', 'gemini-2.5-pro'] {allow-input: true}

service_key = 'vmchat_' + secrets.token_urlsafe(24)

os.environ['LLM_PROVIDER'] = 'langchain'
os.environ['LLM_BASE_URL'] = 'https://generativelanguage.googleapis.com/v1beta/openai/'
os.environ['LLM_API_KEY'] = GEMINI_API_KEY
os.environ['LLM_MODEL'] = MODEL_ID
# Gemini OpenAI compatibility 下，text-json 比严格 structured output 更稳妥
os.environ['LLM_OUTPUT_MODE'] = 'text-json'
os.environ['SERVICE_API_KEY'] = service_key
# 临时联调允许本地任意 origin，浏览器不携带 cookie，只使用 Bearer token
os.environ['CORS_ORIGINS'] = '*'
os.environ['VMCHAT_SQL_KNOWLEDGE_PATH'] = 'delivery/02_vm_modules_sql_statements.md'
os.environ['MAX_SKILL_RESOURCE_READS'] = '24'
os.environ['MAX_PROMPT_CHARS'] = '120000'

print('✅ vmChat 配置完成')
print('模型:', MODEL_ID)
print('Gemini Key: 已从 Secret/密码输入框加载，不会打印')


In [ ]:
# 3) 先单独验证 Gemini OpenAI-compatible 调用
from app.provider.langchain_provider import LangChainVmChatProvider

provider = LangChainVmChatProvider(
    model_name=os.environ['LLM_MODEL'],
    base_url=os.environ['LLM_BASE_URL'],
    api_key=os.environ['LLM_API_KEY'],
    mode='text-json',
)
model = provider.build_model()
reply = await model.ainvoke([
    {'role': 'system', 'content': '只回答 OK'},
    {'role': 'user', 'content': '连通性测试'}
])
print('✅ Gemini 连通成功:', str(reply.content)[:200])


In [ ]:
# 4) 启动 FastAPI
import subprocess, time, requests, pathlib

for name in ('vmchat_uvicorn', 'vmchat_cloudflared'):
    proc = globals().get(name)
    if proc and proc.poll() is None:
        proc.terminate()

uvicorn_log_path = '/content/vmchat-uvicorn.log'
uvicorn_log = open(uvicorn_log_path, 'w')
vmchat_uvicorn = subprocess.Popen(
    ['uv', 'run', 'uvicorn', 'app.main:app', '--host', '0.0.0.0', '--port', '8000'],
    stdout=uvicorn_log,
    stderr=subprocess.STDOUT,
    env=os.environ.copy(),
)

for _ in range(90):
    try:
        r = requests.get('http://127.0.0.1:8000/health', timeout=2)
        if r.status_code == 200:
            print('✅ FastAPI 已启动:', r.json())
            break
    except Exception:
        pass
    time.sleep(1)
else:
    print(pathlib.Path(uvicorn_log_path).read_text(errors='ignore')[-8000:])
    raise RuntimeError('FastAPI 启动失败')


In [ ]:
# 5) 创建 Cloudflare 临时 HTTPS Tunnel
import os, re, stat, subprocess, time, pathlib

cloudflared = '/content/cloudflared'
if not os.path.exists(cloudflared):
    !wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    os.chmod(cloudflared, os.stat(cloudflared).st_mode | stat.S_IEXEC)

tunnel_log_path = '/content/cloudflared.log'
tunnel_log = open(tunnel_log_path, 'w')
vmchat_cloudflared = subprocess.Popen(
    [cloudflared, 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
    stdout=tunnel_log,
    stderr=subprocess.STDOUT,
)

public_url = None
for _ in range(90):
    time.sleep(1)
    tunnel_log.flush()
    text = pathlib.Path(tunnel_log_path).read_text(errors='ignore')
    match = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', text)
    if match:
        public_url = match.group(0)
        break

if not public_url:
    print(text[-8000:])
    raise RuntimeError('Cloudflare Tunnel 启动失败')

print('✅ Colab Python 公网地址:', public_url)
print('Health:', public_url + '/health')


In [ ]:
# 6) 验证公网地址，并生成本地前端连接配置
import requests, json

health = requests.get(public_url + '/health', timeout=30)
print('公网 health:', health.status_code, health.text[:500])

browser_js = f'''localStorage.setItem('fof-research-hermes-base-url', '{public_url}');
localStorage.setItem('fof-research-hermes-api-key', '{service_key}');
localStorage.removeItem('fof-research-hermes-run-events-disabled');
location.reload();'''

print('\n' + '=' * 72)
print('把下面 4 行复制到你本地前端页面的浏览器 Console：')
print('=' * 72)
print(browser_js)
print('=' * 72)
print('\n你的本地前端继续 npm run dev 即可。')
print('业务接口仍走 /pfl -> http://192.168.1.203:8010/pflfofrest/')
print('AI 请求则直接走上面的 Colab HTTPS 地址。')


## 本地前端运行方式

在你的 Mac 项目目录执行：

```bash
cd web
npm run dev
```

打开 `http://localhost:9528`，然后把上一个单元格输出的 JavaScript 粘贴到浏览器开发者工具 Console。

最终链路：

```text
本地 Vue (localhost:9528)
   ├─ /pfl ─────────────→ 192.168.1.203:8010/pflfofrest
   └─ AI HTTPS ─────────→ trycloudflare.com → Colab FastAPI → Gemini API
```

不要关闭 Colab Runtime；Runtime 被回收后 Tunnel 地址会失效。重新 Run all 后会生成新地址，再把新的 Console 配置粘一次即可。


In [ ]:
# 7) 可选：查看服务日志
from pathlib import Path
print('--- uvicorn ---')
print(Path('/content/vmchat-uvicorn.log').read_text(errors='ignore')[-5000:])
print('\n--- cloudflared ---')
print(Path('/content/cloudflared.log').read_text(errors='ignore')[-3000:])


In [ ]:
# 8) 不用了再运行这个单元格停止临时服务
for name in ('vmchat_cloudflared', 'vmchat_uvicorn'):
    proc = globals().get(name)
    if proc and proc.poll() is None:
        proc.terminate()
print('已停止 Colab 临时服务。')
